<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Conteo_de_Veh%C3%ADculos_con_Yolo_color.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics
!pip install supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 391.6/391.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 9.6 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv
from collections import defaultdict
from google.colab.patches import cv2_imshow

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

VIDEO_PATH = "video1.mp4"

MODEL = "yolo26n.pt"

OUTPUT_PATH = "resultado_vehiculos_bytetrack.mp4"

CONFIDENCE = 0.40

In [ ]:
# ============================================================
# VEHÍCULOS QUE QUEREMOS CONTAR
# ============================================================

CLASSES_TO_COUNT = {
    2: "Auto",
    3: "Motocicleta",
    5: "Autobus",
    7: "Camion"
}


In [ ]:
# ============================================================
# REGIÓN DE INTERÉS
# ============================================================

REGION = np.array([
    [500, 150],
    [1800, 150],
    [1800, 650],
    [500, 650]
], dtype=np.int32)

In [ ]:
# ============================================================
# PARÁMETROS DE ESTABILIDAD
# ============================================================

# Frames consecutivos que un objeto debe estar
# dentro de la región antes de considerarlo válido
MIN_FRAMES_TO_CONFIRM = 10


# Número de frames que ByteTrack conserva
# un objeto cuando temporalmente no lo detecta
TRACK_BUFFER = 60

In [ ]:
# ============================================================
# MODELO
# ============================================================

print("Cargando YOLO...")

model = YOLO(MODEL)

print("Modelo cargado.")

Cargando YOLO...
Modelo cargado.


In [ ]:
# ============================================================
# BYTE TRACK
# ============================================================

tracker = sv.ByteTrack(
    track_activation_threshold=0.25,
    lost_track_buffer=TRACK_BUFFER,
    minimum_matching_threshold=0.8,
    frame_rate=30
)

/tmp/ipykernel_822/2498558552.py:5: FutureWarning: The `ByteTrack` was deprecated since v0.28.0. It will be removed in v0.31.0.
  tracker = sv.ByteTrack(


In [ ]:
# ============================================================
# VIDEO
# ============================================================

cap = cv2.VideoCapture(
    VIDEO_PATH
)

if not cap.isOpened():

    raise Exception(
        "No se pudo abrir el video"
    )

In [ ]:
# ============================================================
# INFORMACIÓN DEL VIDEO
# ============================================================

fps = cap.get(
    cv2.CAP_PROP_FPS
)

width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

total_frames = int(
    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)


print()
print("Información del video")
print("---------------------")
print(f"Resolución: {width}x{height}")
print(f"FPS: {fps:.2f}")
print(f"Frames: {total_frames}")


Información del video
---------------------
Resolución: 1920x1080
FPS: 25.00
Frames: 1501


In [ ]:
# ============================================================
# VIDEO DE SALIDA
# ============================================================

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (width, height)
)

In [ ]:
# ============================================================
# VARIABLES DEL CONTADOR
# ============================================================

# IDs que ya fueron contados
counted_ids = set()

# Cuántos frames lleva cada ID dentro de la región
inside_frames = defaultdict(int)

# Última posición conocida
last_positions = {}

# Clase asociada al ID
track_classes = {}

# Conteo por clase
total_count = defaultdict(int)

# Objetos actualmente dentro
inside_ids = set()

# ------------------------------------------------------------
# CONTEO POR COLOR
# ------------------------------------------------------------

COLOR_NAMES = [
    "Blanco",
    "Negro",
    "Gris",
    "Rojo",
    "Azul",
    "Verde",
    "Amarillo",
    "Naranja",
    "Cafe"
]

# Conteo final por color
color_count = defaultdict(int)

# Votos de color por cada tracker_id.
# Se acumulan varios frames para hacer el color más estable.
track_color_votes = defaultdict(lambda: defaultdict(int))

# Color definitivo asignado a cada vehículo
track_colors = {}


In [ ]:
# ============================================================
# FUNCIÓN ROI
# ============================================================

def point_inside_region(
    point,
    polygon
):

    x, y = point

    return cv2.pointPolygonTest(
        polygon,
        (
            float(x),
            float(y)
        ),
        False
    ) >= 0


In [ ]:
# ============================================================
# DASHBOARD
# ============================================================

def draw_dashboard(
    frame,
    total_count,
    color_count,
    inside_ids,
    counted_ids
):

    h, w = frame.shape[:2]

    dashboard_height = 180

    overlay = frame.copy()

    cv2.rectangle(
        overlay,
        (0, 0),
        (w, dashboard_height),
        (15, 15, 15),
        -1
    )

    frame = cv2.addWeighted(
        overlay,
        0.85,
        frame,
        0.15,
        0
    )

    # --------------------------------------------------------
    # TÍTULO
    # --------------------------------------------------------

    cv2.putText(
        frame,
        "VEHICLE TRACKING + COLOR",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    # --------------------------------------------------------
    # TOTAL / DENTRO / IDS
    # --------------------------------------------------------

    total = sum(total_count.values())

    cv2.putText(
        frame,
        f"TOTAL: {total}",
        (20, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"DENTRO: {len(inside_ids)}",
        (220, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    cv2.putText(
        frame,
        f"IDS: {len(counted_ids)}",
        (440, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 180, 0),
        2
    )

    # --------------------------------------------------------
    # CLASES
    # --------------------------------------------------------

    cv2.putText(
        frame,
        "VEHICULOS:",
        (20, 105),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (200, 200, 200),
        1
    )

    x = 130

    for clase, cantidad in total_count.items():

        texto = f"{clase}: {cantidad}"

        cv2.putText(
            frame,
            texto,
            (x, 105),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            1
        )

        x += 155

    # --------------------------------------------------------
    # COLORES
    # --------------------------------------------------------

    cv2.putText(
        frame,
        "COLORES:",
        (20, 140),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (200, 200, 200),
        1
    )

    x = 130

    for color_name in COLOR_NAMES:

        cantidad = color_count.get(color_name, 0)

        texto = f"{color_name}: {cantidad}"

        cv2.putText(
            frame,
            texto,
            (x, 140),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.48,
            (255, 255, 255),
            1
        )

        x += 125

        # Segunda línea si no cabe
        if x > w - 120:
            x = 130
            y_color = 165
            break
    else:
        y_color = 165

    # Mostrar colores restantes en segunda línea cuando sea necesario
    if len(COLOR_NAMES) > 1 and x == 130:
        start_index = 0
        # La primera línea ya mostró hasta el último que cupo.
        # Redibujamos de forma controlada para evitar desbordamiento.
        cv2.rectangle(
            frame,
            (0, 145),
            (w, dashboard_height),
            (15, 15, 15),
            -1
        )

        cv2.putText(
            frame,
            "COLORES:",
            (20, 165),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (200, 200, 200),
            1
        )

        x = 130

        for color_name in COLOR_NAMES:

            cantidad = color_count.get(color_name, 0)

            cv2.putText(
                frame,
                f"{color_name}: {cantidad}",
                (x, 165),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.43,
                (255, 255, 255),
                1
            )

            x += 110

            if x > w - 100:
                break

    return frame


In [ ]:
# ============================================================
# PROCESAMIENTO
# ============================================================

frame_number = 0

print()
print("Procesando video...")
print()


def detect_vehicle_color(crop):
    """Estima el color predominante del vehículo usando HSV."""

    if crop is None or crop.size == 0:
        return None

    h, w = crop.shape[:2]

    # Evitar bordes del bounding box y, en lo posible,
    # concentrarnos en la carrocería central.
    margin_x = int(w * 0.15)
    margin_y = int(h * 0.20)

    if w > 20 and h > 20:
        crop = crop[
            margin_y:max(margin_y + 1, h - margin_y),
            margin_x:max(margin_x + 1, w - margin_x)
        ]

    if crop.size == 0:
        return None

    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

    H = hsv[:, :, 0]
    S = hsv[:, :, 1]
    V = hsv[:, :, 2]

    # Ignorar píxeles extremadamente oscuros y reflejos extremos.
    valid = V > 30

    if np.count_nonzero(valid) < 20:
        return "Negro"

    h_values = H[valid]
    s_values = S[valid]
    v_values = V[valid]

    median_s = np.median(s_values)
    median_v = np.median(v_values)

    # Negro
    if median_v < 65:
        return "Negro"

    # Blanco
    if median_s < 35 and median_v > 170:
        return "Blanco"

    # Gris
    if median_s < 45:
        return "Gris"

    # Color cromático: usamos la mediana del tono.
    hue = float(np.median(h_values))

    if hue < 8 or hue >= 170:
        return "Rojo"
    elif hue < 18:
        return "Naranja"
    elif hue < 35:
        return "Amarillo"
    elif hue < 85:
        return "Verde"
    elif hue < 135:
        return "Azul"
    else:
        return "Rojo"


while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # ========================================================
    # YOLO
    # ========================================================

    results = model(
        frame,
        conf=CONFIDENCE,
        verbose=False
    )[0]

    # ========================================================
    # SUPERVISION
    # ========================================================

    detections = sv.Detections.from_ultralytics(results)

    # ========================================================
    # FILTRAR VEHÍCULOS
    # ========================================================

    if len(detections) > 0:

        mask = np.array([
            class_id in CLASSES_TO_COUNT
            for class_id in detections.class_id
        ])

        detections = detections[mask]

    # ========================================================
    # BYTE TRACK
    # ========================================================

    detections = tracker.update_with_detections(detections)

    current_inside_ids = set()

    # ========================================================
    # PROCESAR TRACKS
    # ========================================================

    for i in range(len(detections)):

        xyxy = detections.xyxy[i]

        class_id = int(detections.class_id[i])

        tracker_id = int(detections.tracker_id[i])

        x1, y1, x2, y2 = map(int, xyxy)

        # Limitar coordenadas al frame
        x1 = max(0, min(x1, width - 1))
        y1 = max(0, min(y1, height - 1))
        x2 = max(0, min(x2, width))
        y2 = max(0, min(y2, height))

        # ====================================================
        # CENTRO INFERIOR
        # ====================================================

        center_x = int((x1 + x2) / 2)
        center_y = int(y2)

        center = (center_x, center_y)

        last_positions[tracker_id] = center
        track_classes[tracker_id] = class_id

        # ====================================================
        # ¿ESTÁ DENTRO DEL ROI?
        # ====================================================

        inside = point_inside_region(center, REGION)

        if inside:

            current_inside_ids.add(tracker_id)

            inside_frames[tracker_id] += 1

            # ------------------------------------------------
            # ESTIMAR COLOR DURANTE VARIOS FRAMES
            # ------------------------------------------------

            crop = frame[y1:y2, x1:x2]

            detected_color = detect_vehicle_color(crop)

            if detected_color is not None:

                track_color_votes[
                    tracker_id
                ][detected_color] += 1

            # ------------------------------------------------
            # CONTAR SOLO CUANDO ESTÁ CONFIRMADO
            # ------------------------------------------------

            if inside_frames[tracker_id] >= MIN_FRAMES_TO_CONFIRM:

                if tracker_id not in counted_ids:

                    counted_ids.add(tracker_id)

                    class_name = CLASSES_TO_COUNT[class_id]

                    total_count[class_name] += 1

                    # ----------------------------------------
                    # COLOR DEFINITIVO
                    # ----------------------------------------

                    votes = track_color_votes[tracker_id]

                    if votes:
                        final_color = max(
                            votes,
                            key=votes.get
                        )
                    else:
                        final_color = "Desconocido"

                    track_colors[tracker_id] = final_color

                    color_count[final_color] += 1

        # ====================================================
        # COLOR DEL VEHÍCULO PARA LA ETIQUETA
        # ====================================================

        vehicle_color = track_colors.get(
            tracker_id,
            None
        )

        if vehicle_color is None:

            votes = track_color_votes.get(
                tracker_id,
                {}
            )

            if votes:
                vehicle_color = max(
                    votes,
                    key=votes.get
                )
            else:
                vehicle_color = "..."

        # ====================================================
        # COLOR DEL BOUNDING BOX
        # ====================================================

        if tracker_id in counted_ids:
            box_color = (0, 255, 0)
        elif inside:
            box_color = (0, 255, 255)
        else:
            box_color = (0, 165, 255)

        # ====================================================
        # BOUNDING BOX
        # ====================================================

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            box_color,
            2
        )

        # ====================================================
        # LABEL
        # ====================================================

        class_name = CLASSES_TO_COUNT[class_id]

        label = (
            f"{class_name} "
            f"ID:{tracker_id} "
            f"{vehicle_color}"
        )

        cv2.putText(
            frame,
            label,
            (
                x1,
                max(y1 - 10, 20)
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.50,
            box_color,
            2
        )

        # ====================================================
        # CENTRO
        # ====================================================

        cv2.circle(
            frame,
            center,
            5,
            box_color,
            -1
        )

    # ========================================================
    # ACTUALIZAR OBJETOS DENTRO
    # ========================================================

    inside_ids = current_inside_ids

    # ========================================================
    # DIBUJAR ROI
    # ========================================================

    cv2.polylines(
        frame,
        [REGION],
        True,
        (255, 0, 255),
        3
    )

    x_roi, y_roi = REGION[0]

    cv2.putText(
        frame,
        "REGION DE CONTEO",
        (
            x_roi,
            max(y_roi - 10, 20)
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 0, 255),
        2
    )

    # ========================================================
    # DASHBOARD
    # ========================================================

    frame = draw_dashboard(
        frame,
        total_count,
        color_count,
        inside_ids,
        counted_ids
    )

    # ========================================================
    # GUARDAR FRAME
    # ========================================================

    out.write(frame)

    # ========================================================
    # PROGRESO
    # ========================================================

    if frame_number % 100 == 0:

        progress = (
            frame_number /
            total_frames
        ) * 100

        print(
            f"Procesado: "
            f"{frame_number}/"
            f"{total_frames} "
            f"({progress:.1f}%)"
        )

# ============================================================
# CERRAR
# ============================================================

cap.release()
out.release()



Procesando video...

Procesado: 100/1501 (6.7%)
Procesado: 200/1501 (13.3%)
Procesado: 300/1501 (20.0%)
Procesado: 400/1501 (26.6%)
Procesado: 500/1501 (33.3%)
Procesado: 600/1501 (40.0%)
Procesado: 700/1501 (46.6%)
Procesado: 800/1501 (53.3%)
Procesado: 900/1501 (60.0%)
Procesado: 1000/1501 (66.6%)
Procesado: 1100/1501 (73.3%)
Procesado: 1200/1501 (79.9%)
Procesado: 1300/1501 (86.6%)
Procesado: 1400/1501 (93.3%)
Procesado: 1500/1501 (99.9%)


In [ ]:
# ============================================================
# RESULTADO
# ============================================================

print()
print("=" * 55)
print("             RESULTADO FINAL")
print("=" * 55)
print()

print(
    f"Total vehículos: "
    f"{sum(total_count.values())}"
)

print()
print("CONTEO POR TIPO")
print("----------------")

for clase, cantidad in total_count.items():
    print(f"{clase}: {cantidad}")

print()
print("CONTEO POR COLOR")
print("----------------")

for color_name in COLOR_NAMES:
    cantidad = color_count.get(color_name, 0)
    print(f"{color_name}: {cantidad}")

print()
print(
    f"IDs contabilizados: "
    f"{len(counted_ids)}"
)

print()
print("COLOR ASIGNADO A CADA VEHÍCULO")
print("-------------------------------")

for tracker_id in sorted(track_colors):
    clase_id = track_classes.get(tracker_id)
    clase = CLASSES_TO_COUNT.get(clase_id, "Desconocido")
    color = track_colors[tracker_id]
    print(
        f"ID {tracker_id}: {clase} - {color}"
    )

print()
print(
    f"Video generado: "
    f"{OUTPUT_PATH}"
)

print("=" * 55)



             RESULTADO FINAL

Total vehículos: 34

CONTEO POR TIPO
----------------
Auto: 31
Camion: 2
Motocicleta: 1

CONTEO POR COLOR
----------------
Blanco: 4
Negro: 0
Gris: 8
Rojo: 3
Azul: 19
Verde: 0
Amarillo: 0
Naranja: 0
Cafe: 0

IDs contabilizados: 34

COLOR ASIGNADO A CADA VEHÍCULO
-------------------------------
ID 4: Auto - Azul
ID 5: Auto - Azul
ID 6: Auto - Azul
ID 8: Auto - Gris
ID 9: Auto - Rojo
ID 10: Auto - Blanco
ID 11: Auto - Azul
ID 14: Auto - Gris
ID 15: Auto - Gris
ID 19: Auto - Azul
ID 21: Auto - Gris
ID 23: Auto - Azul
ID 25: Auto - Azul
ID 27: Auto - Azul
ID 28: Auto - Rojo
ID 30: Auto - Azul
ID 31: Auto - Azul
ID 32: Auto - Azul
ID 33: Auto - Rojo
ID 34: Camion - Gris
ID 35: Auto - Azul
ID 36: Auto - Azul
ID 38: Auto - Gris
ID 39: Auto - Gris
ID 40: Motocicleta - Gris
ID 43: Auto - Azul
ID 44: Auto - Azul
ID 45: Auto - Azul
ID 47: Auto - Azul
ID 48: Auto - Azul
ID 51: Auto - Azul
ID 52: Auto - Blanco
ID 53: Auto - Blanco
ID 54: Auto - Blanco

Video generado:

In [ ]:
from google.colab import files

# ============================================================
# DESCARGAR
# ============================================================

files.download(
    OUTPUT_PATH
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>